# 📊 Notebook 02 — Exploratory Data Analysis
### Walmart Retail Sales Demand Forecasting

---

## 🎯 Goal

Before building any model, we need to *see* the data. This notebook answers:

- How does sales change over time? Is there a seasonal pattern?
- Do holiday weeks actually sell more?
- How different are stores from each other?
- Do economic factors (temperature, fuel price, CPI, unemployment) visibly affect sales?

The answers will tell us what our forecasting model needs to capture.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('../data/walmart_sales.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

print(f'Loaded {len(df):,} rows across {df["Store"].nunique()} stores.')

## 2. Variable Distributions

Let's look at the shape of each variable — is it normally distributed? Skewed? Are there outliers?


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Weekly Sales distribution
sns.histplot(df['Weekly_Sales'] / 1e6, kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Distribution of Weekly Sales')
axes[0, 0].set_xlabel('Sales ($ Millions)')

# Store distribution (all 45 stores appear equally)
sns.countplot(x='Store', data=df, ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Records per Store (should be equal)')
axes[0, 1].tick_params(axis='x', rotation=90)
axes[0, 1].set_xlabel('')

# Holiday flag
counts = df['Holiday_Flag'].value_counts()
axes[1, 0].bar(['Normal week (0)', 'Holiday week (1)'], counts.values, color=['steelblue', 'coral'])
axes[1, 0].set_title('Holiday vs Normal Weeks')
for i, v in enumerate(counts.values):
    axes[1, 0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Temperature distribution
sns.histplot(df['Temperature'], kde=True, ax=axes[1, 1], color='steelblue')
axes[1, 1].set_title('Distribution of Temperature (°F)')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.histplot(df['Fuel_Price'],    kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Fuel Price ($/gallon)')

sns.histplot(df['CPI'],           kde=True, ax=axes[1], color='steelblue')
axes[1].set_title('CPI (Consumer Price Index)')

sns.histplot(df['Unemployment'],  kde=True, ax=axes[2], color='steelblue')
axes[2].set_title('Unemployment Rate (%)')

plt.tight_layout()
plt.show()

### 💡 What we notice

- **Weekly Sales** is right-skewed — most stores sell in the $500K–$2M range, but a handful of large stores push much higher
- **Holiday weeks** make up only ~7% of all records — rare, but potentially high-impact
- **Temperature** is bimodal, suggesting stores are split across warm and cool climates
- **CPI** shows two distinct clusters — stores likely span different inflation-regime regions
- **Unemployment** is spread around 7–9%, reflecting the 2010–2012 post-recession period


## 3. Sales Trend Over Time

How have sales moved across the full 2010–2012 period? Are there visible seasonal patterns?


In [ ]:
# Aggregate total weekly sales across all stores
weekly_total = df.groupby('Date')['Weekly_Sales'].sum() / 1e6  # in millions

plt.figure(figsize=(15, 5))
plt.plot(weekly_total.index, weekly_total.values, color='steelblue', linewidth=1.2)
plt.title('Total Weekly Sales Across All 45 Stores (2010–2012)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Total Sales ($ Millions)')

# Highlight December holiday spikes
for year in [2010, 2011, 2012]:
    plt.axvspan(pd.Timestamp(f'{year}-11-20'), pd.Timestamp(f'{year}-12-31'),
                alpha=0.15, color='coral', label='Nov–Dec holiday window' if year == 2010 else '')

plt.legend()
plt.tight_layout()
plt.show()

### 💡 Key observation

There are **clear, repeating spikes every November–December** — this is the holiday (Thanksgiving + Christmas) effect. Sales jump significantly in late Q4 every year and return to baseline in January.

This is the core seasonality our model needs to learn. A model that doesn't know "December is special" will be wrong every year.


## 4. Do Holiday Weeks Actually Sell More?

The `Holiday_Flag` column marks weeks with major holidays. Let's verify whether this actually translates to higher sales.


In [ ]:
# Compare average sales: holiday vs normal weeks
holiday_avg    = df[df['Holiday_Flag'] == 1]['Weekly_Sales'].mean()
non_holiday_avg = df[df['Holiday_Flag'] == 0]['Weekly_Sales'].mean()
uplift = (holiday_avg - non_holiday_avg) / non_holiday_avg * 100

print(f'Average sales — normal week  : ${non_holiday_avg:>10,.0f}')
print(f'Average sales — holiday week : ${holiday_avg:>10,.0f}')
print(f'Holiday uplift               : +{uplift:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar comparison
bars = axes[0].bar(['Normal week', 'Holiday week'],
                   [non_holiday_avg / 1e6, holiday_avg / 1e6],
                   color=['steelblue', 'coral'], width=0.5)
axes[0].set_title('Average Weekly Sales: Normal vs Holiday')
axes[0].set_ylabel('Average Sales ($ Millions)')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'${bar.get_height():.2f}M', ha='center', fontweight='bold')

# Distribution comparison
sns.kdeplot(data=df[df['Holiday_Flag'] == 0],  x='Weekly_Sales', ax=axes[1],
            label='Normal week', fill=True, alpha=0.4, color='steelblue')
sns.kdeplot(data=df[df['Holiday_Flag'] == 1],  x='Weekly_Sales', ax=axes[1],
            label='Holiday week', fill=True, alpha=0.4, color='coral')
axes[1].set_title('Sales Distribution: Normal vs Holiday Weeks')
axes[1].set_xlabel('Weekly Sales ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

### 💡 Key insight

Holiday weeks generate **noticeably higher sales** — this confirms that `Holiday_Flag` is a meaningful signal. Any model that ignores holidays will consistently under-predict during those weeks, leading to stockouts at the worst possible time (when customer traffic is highest).


## 5. How Different Are Individual Stores?

Before deciding whether to build one model for all stores or separate models per store, we need to see how much stores vary.


In [ ]:
# Average weekly sales per store, sorted from highest to lowest
store_avg = df.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=False)

plt.figure(figsize=(16, 5))
bars = plt.bar(store_avg.index.astype(str), store_avg.values / 1e6, color='steelblue')

# Highlight top 3 and bottom 3
for i, (store, val) in enumerate(store_avg.items()):
    if i < 3:
        bars[i].set_color('coral')
    elif i >= len(store_avg) - 3:
        bars[i].set_color('lightsteelblue')

plt.title('Average Weekly Sales by Store (sorted)', fontsize=13)
plt.xlabel('Store ID')
plt.ylabel('Avg Weekly Sales ($ Millions)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f'\nHighest-selling store : Store {store_avg.index[0]}  — avg ${store_avg.iloc[0]:>8,.0f}/week')
print(f'Lowest-selling store  : Store {store_avg.index[-1]} — avg ${store_avg.iloc[-1]:>8,.0f}/week')
print(f'Ratio (highest/lowest): {store_avg.iloc[0] / store_avg.iloc[-1]:.1f}×')

### 💡 Key insight

The top stores sell **several times more** than the bottom stores. A single global model would struggle to handle this — it would overfit to the large stores and fail on the small ones. This strongly suggests we should build **one model per store**.


## 6. Do External Factors Affect Sales?

We have temperature, fuel price, CPI, and unemployment. Do any of these visibly correlate with sales?


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

vars_to_check = [
    ('Temperature',  'Temperature (°F)'),
    ('Fuel_Price',   'Fuel Price ($/gallon)'),
    ('CPI',          'CPI (Consumer Price Index)'),
    ('Unemployment', 'Unemployment Rate (%)'),
]

for ax, (col, label) in zip(axes.flat, vars_to_check):
    ax.scatter(df[col], df['Weekly_Sales'] / 1e6, alpha=0.3, s=8, color='steelblue')
    ax.set_xlabel(label)
    ax.set_ylabel('Weekly Sales ($ Millions)')
    ax.set_title(f'{label} vs Weekly Sales')

plt.tight_layout()
plt.show()

In [ ]:
# Compute correlations with Weekly_Sales
numeric_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
correlations = df[numeric_cols + ['Weekly_Sales']].corr()['Weekly_Sales'].drop('Weekly_Sales')

print('Correlation with Weekly_Sales:')
for col, corr in correlations.sort_values(key=abs, ascending=False).items():
    direction = 'positive' if corr > 0 else 'negative'
    print(f'  {col:<15}: {corr:+.3f}  ({direction})')

### 💡 Key insight

The scatter plots show a lot of vertical spread — sales vary enormously even at the same temperature or fuel price. The correlations are weak, which tells us:

- These economic variables **alone** are not strong predictors of sales
- The dominant driver is **which store it is** and **what time of year it is** (seasonality)
- That said, these variables could still improve accuracy as additional signals in a richer model


## ✅ EDA Summary — What We Learned

| Finding | Implication for Modelling |
|---|---|
| Strong holiday spikes every Nov–Dec | Model must understand seasonality |
| Holiday weeks sell ~7% more on average | `Holiday_Flag` should be a feature |
| Stores vary dramatically in sales volume | Build one model **per store** |
| Economic variables have weak direct correlation | Useful as supplementary features, not primary drivers |
| No missing data | No imputation needed |

**Next:** Notebook 03 will build three forecasting models — from simple baselines to Facebook's Prophet — and measure how accurate each one is.
